## Library management System

In [42]:
import psycopg2
from psycopg2.extras import RealDictCursor
from datetime import date
import pandas as pd

In [ ]:
class library():
    def __init__(self):
        self.conn = psycopg2.connect(
            host="localhost",
            port="5432",
            dbname="",
            user="postgres",
            password=""
        )
        self.cur = self.conn.cursor(cursor_factory=RealDictCursor)

    def authentication(self, user_choice):
        self.user_choice = user_choice
        self.cur.execute('SELECT * FROM student WHERE s_id = %s', (user_choice,))
        self.row = self.cur.fetchone()
        if self.row:
            print('Student exists.\n', '--'*40)
            return True
        else:
            print(f'\nThis id {user_choice} did not exist ...\n', '--'*40)
            return False


    def show_all(self):
        self.cur.execute('SELECT * FROM book')
        all_books = self.cur.fetchall()

        if not all_books:
            print('No books found in the library.')
            return

        print('\nLibrary has the following books:\n')
        print(f"{'ID':<5}{'Book Name':<25}{'Author':<25}{'Genre':<15}{'Qty':<5}")
        print('-' * 75)

        for book in all_books:
            print(f"{book['b_id']:<5}{book['b_name']:<25}{book['b_author']:<25}{book['b_genre']:<15}{book['b_quantity']:<5}")

        print('-' * 75)


    def search(self):
        user_type = input('How you want to search : \n 1. Book name \n 2. Author name \n 3. Genre\n')
        book_info = input('Enter about it : ')

        if user_type == '1':
            self.type = 'b_name'
        elif user_type == '2':
            self.type = 'b_author'
        elif user_type == '3':
            self.type = 'b_genre'
        else:
            print('\nInvalid choice ...\n', '--'*40)
            return False

        self.cur.execute(f'SELECT * FROM book WHERE {self.type} ILIKE %s', (f'%{book_info}%',))
        self.book_row = self.cur.fetchone()

        if self.book_row is None:
            print('\nNo matching book found ...\n', '--'*40)
            return False

        print(self.book_row, '\n', '--'*40)
        return True

    def issue(self):
        if not self.search():
            print('This book does not exist in our collection.')
            return

        b_id = self.book_row['b_id']
        s_id = self.row['s_id']

        try:
            self.cur.execute(
                'UPDATE book SET b_quantity = b_quantity - 1 '
                'WHERE b_id = %s AND b_quantity > 0 '
                'RETURNING b_quantity',
                (b_id,))
            result = self.cur.fetchone()

            if result is None:
                print('This book is issued to someone else and has no more copies.')
                self.conn.rollback()
                return

            self.cur.execute('INSERT INTO issue (s_id, b_id, issue_date) VALUES (%s, %s, %s)',(s_id, b_id, date.today()))

            self.conn.commit()
            print(f'Book issued to {self.row["s_name"]} (id {s_id}). Remaining copies: {result["b_quantity"]}')

        except Exception as e:
            self.conn.rollback()
            print(f'Something went wrong, transaction cancelled: {e}')


    def book_return(self):
        self.book_id = input('Enter book id : ')

        self.cur.execute('SELECT * FROM book WHERE b_id = %s', (self.book_id,))
        book_row = self.cur.fetchone()

        if book_row is None:
            print('This book did not belong to our library ...\n', '--'*40)
            return

        quantity = book_row['b_quantity']
        self.cur.execute('UPDATE book SET b_quantity = %s WHERE b_id = %s', (quantity + 1, self.book_id))

        print(f'Book from {self.row["s_name"]} having id {self.row["s_id"]} has been returned.')

        self.cur.execute(
            'INSERT INTO return_back (issue_id, return_date) '
            'SELECT issue_id, %s FROM issue WHERE s_id = %s AND b_id = %s '
            'ORDER BY issue_date DESC LIMIT 1',
            (date.today(), self.row['s_id'], self.book_id))

        self.conn.commit()
        self.fine()

    def fine(self):
        self.cur.execute('SELECT issue_date FROM issue WHERE s_id = %s ORDER BY issue_date DESC LIMIT 1',(self.row['s_id'],))
        issue_row = self.cur.fetchone()

        if issue_row is None:
            print('No issue record found for this student.')
            return

        issue_date = issue_row['issue_date']
        return_date = date.today()

        late_days = (return_date - issue_date).days

        if late_days > 30:
            fine_amount = late_days * 150
            print(f"You are late for returning book, you have to pay the fine of {fine_amount} Rupees")
            self.cur.execute('UPDATE student SET fine = %s WHERE s_id = %s', (fine_amount, self.row['s_id']))
        else:
            print('You have returned the book on time.')

        self.conn.commit()

In [44]:
obj1 = library()
user_id = input('Enter your id first : ')
user_auth=obj1.authentication(user_id)
while True:
    if user_auth:
        user_choice = input('what you want to do : \n 1. See All Books \n 2. Search Book \n 3. Book Issue \n 4. Book Return \n 5. Exit\n')
        if user_choice == '1':
            obj1.show_all()
        elif user_choice=='2':
            obj1.search()
        elif user_choice == '3':
            obj1.issue()
        elif user_choice == '4':
            obj1.book_return()
        elif user_choice == '5':
            break
        else:
            print('Invalid choice , select from within range ...')
    else:
        print('Something went wrong while authenticating')
        break

Student exists.
 --------------------------------------------------------------------------------

Library has the following books:

ID   Book Name                Author                   Genre          Qty  
---------------------------------------------------------------------------
1    The Alchemist            Paulo Coelho             Fiction        5    
3    1984                     George Orwell            Dystopian      4    
4    Atomic Habits            James Clear              Self-Help      6    
5    The Hobbit               J.R.R. Tolkien           Fantasy        2    
2    Sapiens                  Yuval Noah Harari        Non-Fiction    3    
---------------------------------------------------------------------------
